## Importing Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import time
import copy

## Selecting device (CPU or GPU)
If GPU is not available then use CPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Defining class names

In [ ]:
class_names = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

## Data transformations

#### Training Transformations
1. Resize images to 224x224
2. Random flip, rotation → adds variation
3. Adjust brightness and contrast
4. Convert image → tensor
5. Normalize values

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

#### Validation Transformations
1. Resize images to 224x224
2. Convert image → tensor
3. Normalize values
(no randomness)

In [ ]:
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

## Loading Datasets

Reads images from folders like:
```
seg_train/
    buildings/
    forest/
```
Automatically assigns labels based on folder names

In [ ]:
train_dataset = datasets.ImageFolder('/content/intel_data/seg_train/seg_train',
                                      transform=train_transforms)
val_dataset   = datasets.ImageFolder('/content/intel_data/seg_test/seg_test',
                                      transform=val_transforms)

## Creating DataLoaders

- Loads data in batches (32 images at a time)
- shuffle=True for training → better learning
- shuffle=False for validation → consistent results

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)

## Printing dataset info

In [ ]:
print(f"Training samples   : {len(train_dataset)}")
print(f"Validation samples : {len(val_dataset)}")
print(f"Classes            : {class_names}")

## Load a pre-trained model

In [ ]:
model = models.vgg16(pretrained=True)

## Freeze all layers

Stops the model from updating its existing weights

Means:
- Keep learned knowledge
- Don’t retrain the whole network (saves time + avoids overfitting)

In [ ]:
for param in model.parameters():
    param.requires_grad = False

## Modify the final layer

Gets number of inputs to the last layer

In [ ]:
num_features = model.classifier[6].in_features

- Reduces features → 256 neurons
- Adds activation (ReLU)
- Adds dropout (prevents overfitting)
- Final output → 6 classes

In [ ]:
model.classifier[6] = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 6)
)

## Move model to device

In [ ]:
model = model.to(device)

## Count trainable parameters

In [ ]:
print("ResNet50 loaded and modified for 6 classes!")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Loss function & optimizer

In [ ]:
# It minimizes the error between these distributions, typically by applying a softmax function followed by log loss to measure accuracy.
criterion = nn.CrossEntropyLoss()

In [ ]:
# An Optimization Algorithm
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

## Learning rate scheduler

After every 5 epochs, learning rate becomes 10x smaller

Helps:
- Fine-tune learning
- Avoid overshooting

In [ ]:
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

## Training setup

In [ ]:
# Store results in history (loss + accuracy)
history = {'train_loss': [], 'val_loss': [],
           'train_acc': [],  'val_acc': []}

best_val_acc = 0.0          # track best performance
EPOCHS = 10                 # model trains 10 times over data
checkpoint_path = '/content/drive/MyDrive/NewsImageCNN/checkpoints/best_model.pth'

## Training loop

In [ ]:
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 40)

    # Two phases: Train & Validation
    for phase in ['train', 'val']:
        # Training phase: Model learn
        if phase == 'train':
            model.train()
            loader = train_loader   # Loading the training dataset to loader
        # Validation Phase: Model only evaluates
        else:
            model.eval()
            loader = val_loader     # Loading the validation dataset to loader

        running_loss = 0.0
        running_corrects = 0

        # Loop through data: Takes batches of images + labels
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                # Forward pass: Model predicts and compare with actual labels
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                _, preds = torch.max(outputs, 1)

                # Backward pass (only in training): Adjusts weights to improve predictions
                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            # Track performance -> total loss, correct predictions
            running_loss     += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        # Update learning rate, Happens after each training epoch
        if phase == 'train':
            scheduler.step()

        epoch_loss = running_loss / len(loader.dataset)
        epoch_acc  = running_corrects.double() / len(loader.dataset)

        print(f"{phase.upper()} — Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f}")

        if phase == 'train':
            history['train_loss'].append(epoch_loss)
            history['train_acc'].append(epoch_acc.item())
        else:
            history['val_loss'].append(epoch_loss)
            history['val_acc'].append(epoch_acc.item())

            # Save best model
            if epoch_acc > best_val_acc:
                best_val_acc = epoch_acc
                torch.save(model.state_dict(), checkpoint_path)
                print(f"  ✅ Best model saved! Val Acc: {best_val_acc:.4f}")

print("\nTraining Complete!")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")

## Load the best trained model

In [ ]:
model.load_state_dict(torch.load(checkpoint_path))      # Loads the best saved weights (highest validation accuracy)
model.eval()        # switches model to evaluation mode (no dropout, no training behavior)

In [ ]:
# Prepare lists to store results
all_preds  = []
all_labels = []

# Turn off gradient calculation, Speeds up evaluation + saves memory
with torch.no_grad():
    # Run model on validation data
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)     # Model gives probabilities for each class
        _, preds = torch.max(outputs, 1)    # Choose the class with highest score
        # Save: Predictions & Actual labels
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

## Generate classification report

In [ ]:
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Plot 1 - Accuracy
axes[0].plot(history['train_acc'], label='Train Acc', marker='o')
axes[0].plot(history['val_acc'],   label='Val Acc',   marker='o')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Plot 2 - Loss
axes[1].plot(history['train_loss'], label='Train Loss', marker='o')
axes[1].plot(history['val_loss'],   label='Val Loss',   marker='o')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

# Plot 3 - Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names, ax=axes[2])
axes[2].set_title('Confusion Matrix')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/NewsImageCNN/results_vgg16.png', dpi=150)
plt.show()
print("Results saved to Drive!")